In [ ]:
import time
import heapq

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def get_successors(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt, new_mt[i]
        
    if c > 0: 
        new_state, cost = swap(mt, pos, pos - 1)
        successors.append(("Trái", new_state, cost))
    if c < 2: 
        new_state, cost = swap(mt, pos, pos + 1)
        successors.append(("Phải", new_state, cost))
    if r > 0: 
        new_state, cost = swap(mt, pos, pos - 3)
        successors.append(("Lên", new_state, cost))
    if r < 2: 
        new_state, cost = swap(mt, pos, pos + 3)
        successors.append(("Xuống", new_state, cost))
    return successors

def count_discrete_inversions(mt):
    """g(n): Đếm số cặp nghịch thế rời rạc (chỉ xét cặp cách nhau >= 2 vị trí)."""
    inv_count = 0
    arr = [x for x in mt if x != 0]
    for i in range(len(arr)):
        # Bắt đầu từ i+2 để chỉ xét các cặp "rời rạc"
        for j in range(i + 2, len(arr)):
            if arr[i] > arr[j]:
                inv_count += 1
    return inv_count

def count_misplaced(mt, goal):
    """h(n): Đếm số ô sai vị trí so với trạng thái đích."""
    misplaced = 0
    for i in range(9):
        if mt[i] != 0 and mt[i] != goal[i]:
            misplaced += 1
    return misplaced

def get_action_name(path):
    if not path:
        return "S"
    mapping = {"Trái": "L", "Phải": "R", "Lên": "U", "Xuống": "D"}
    return "N_" + "".join([mapping[a] for a, _ in path])

def astar_solve(start_state, goal_state, mode="late"):
    """A* Search với hàm đánh giá tuỳ chỉnh:
       f(n) = g(n) + h(n)
       g(n) = số cặp nghịch thế rời rạc (count_discrete_inversions)
       h(n) = số ô sai vị trí (count_misplaced)
    """
    log_data = []
    
    if start_state == goal_state:
        return [], 0, log_data
        
    counter = 0
    frontier = []
    
    g_start = count_discrete_inversions(start_state)
    h_start = count_misplaced(start_state, goal_state)
    f_start = g_start + h_start
    
    heapq.heappush(frontier, (f_start, counter, start_state, []))
    
    explored = {tuple(start_state): f_start}
    state_to_name = {tuple(start_state): "S"}
    nodes_generated = 1
    
    log_data.append({
        "step": 0,
        "action_html": "Khởi tạo nút gốc $S$",
        "frontier_str": f"[S(f={f_start}, g={g_start}, h={h_start})]",
        "reached_str": f"{{S: {f_start}}}"
    })
    
    step_count = 0
    while frontier:
        step_count += 1
        f, _, node, path = heapq.heappop(frontier)
        node_name = get_action_name(path)
        
        if mode == "late" and node == goal_state:
            action_html = f"Bốc nút nhỏ nhất tại Miệng là {node_name} ra xét:<br>Kiểm tra thấy cấu hình trùng khớp hoàn toàn với Goal.<br>👉 THUẬT TOÁN DỪNG VÀ TRẢ VỀ KẾT QUẢ"
            log_data.append({
                "step": step_count,
                "action_html": action_html,
                "frontier_str": "(Giữ nguyên)",
                "reached_str": "(Giữ nguyên)"
            })
            return path, nodes_generated, log_data
            
        if mode == "late" and f > explored.get(tuple(node), float('inf')):
            continue
            
        successors = get_successors(node)
        children_logs = []
        for action, child, _ in successors:
            new_g = count_discrete_inversions(child)
            h = count_misplaced(child, goal_state)
            new_f = new_g + h
            child_tuple = tuple(child)
            
            child_path = path + [(action, child)]
            child_name = get_action_name(child_path)
            
            if child_tuple not in state_to_name:
                state_to_name[child_tuple] = child_name
            
            children_logs.append((action, child_name, new_f, new_g, h))
            
            if mode == "early":
                nodes_generated += 1
                if child == goal_state:
                    return child_path, nodes_generated, log_data
                if child_tuple not in explored or new_f < explored[child_tuple]:
                    explored[child_tuple] = new_f
                    counter += 1
                    heapq.heappush(frontier, (new_f, counter, child, child_path))
            else:
                if child_tuple not in explored or new_f < explored[child_tuple]:
                    explored[child_tuple] = new_f
                    nodes_generated += 1
                    counter += 1
                    heapq.heappush(frontier, (new_f, counter, child, child_path))
                    
        action_html = f"Bốc {node_name} ra mở rộng:<br>"
        action_html += f"Sinh ra {len(successors)} nút con tại Bước {step_count}:<br>"
        for act, cname, cf, cg, ch in children_logs:
            action_html += f"- {cname} ({act}): f = g+h = {cg}+{ch} = <b>{cf}</b><br>"
        
        action_html += "👉 Nạp tất cả và sắp xếp theo f tăng dần."
        
        sorted_frontier = sorted(frontier, key=lambda x: (-x[0], -x[1]))
        frontier_items = []
        for item in sorted_frontier:
            c_name = get_action_name(item[3])
            frontier_items.append(f"{c_name}(f={item[0]})")
        frontier_str = "[" + ", ".join(frontier_items) + "]"
        if sorted_frontier:
            smallest = sorted_frontier[-1]
            s_name = get_action_name(smallest[3])
            frontier_str += f"<br><br>(Nút {s_name} có f={smallest[0]} nhỏ nhất nằm ở Miệng)"
            
        reached_items = []
        for st, f_val in explored.items():
            st_name = state_to_name[st]
            reached_items.append(f"{st_name}: {f_val}")
        reached_str = "{" + ", ".join(reached_items) + "}"
        
        log_data.append({
            "step": step_count,
            "action_html": action_html,
            "frontier_str": frontier_str,
            "reached_str": reached_str
        })
        
    return None, nodes_generated, log_data